In [66]:
import os
import json
import subprocess
from datetime import datetime

In [67]:
# Create folders
os.makedirs("inputs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("sample_repo", exist_ok=True)

# Buggy function
with open("sample_repo/app.py", "w") as f:
    f.write("""
def divide(a, b):
    return a / b
""")

# Test file
with open("sample_repo/test_app.py", "w") as f:
    f.write("""
from app import divide

def test_divide():
    assert divide(10, 0) == 0
""")

# Bug report
with open("inputs/bug_report.txt", "w") as f:
    f.write("""
Title: Division crash on zero input

Expected Behavior:
The function should handle division by zero safely.

Actual Behavior:
The application crashes with ZeroDivisionError.

Environment:
Python 3.11

Steps to Reproduce:
Call divide(10, 0)
""")

# Logs (with noise)
with open("inputs/logs.txt", "w") as f:
    f.write("""
INFO Application started
DEBUG Initializing modules
ERROR ZeroDivisionError: division by zero
Traceback (most recent call last):
  File "app.py", line 2, in divide
    return a / b
INFO Retrying operation
DEBUG Random log line
""")

In [68]:
def extract_errors(log_text):
    return [line for line in log_text.split("\n") if "ERROR" in line or "Traceback" in line]

def run_tests():
    result = subprocess.run(
        ["pytest", "sample_repo", "--color=no"],
        capture_output=True,
        text=True
    )
    return result.stdout

In [69]:
class DebugState:
    def __init__(self):
        self.bug_summary = {}
        self.logs = {}
        self.hypotheses = []
        self.reproduction = {}
        self.fix = {}
        self.review = {}

In [70]:
class TriageAgent:
    def run(self, bug_report, state):
        state.bug_summary = {
            "error_type": "ZeroDivisionError",
            "severity": "Critical",
            "priority": "High",
            "suspected_area": "Arithmetic operation",
            "description": "Crash due to invalid division operation"
        }
        print("[TriageAgent] Completed")
        return state

In [71]:
class LogAgent:
    def run(self, logs, state):
        errors = extract_errors(logs)
        state.logs = {
            "errors": errors,
            "error_count": len(errors),
            "pattern": "ZeroDivisionError detected"
        }
        print("[LogAgent] Completed")
        return state

In [72]:
class HypothesisAgent:
    def run(self, state):
        state.hypotheses = [
            {"cause": "Division by zero", "confidence": 0.9},
            {"cause": "Invalid input handling", "confidence": 0.7}
        ]
        print("[HypothesisAgent] Completed")
        return state

In [73]:
class ReproductionAgent:
    def run(self, state):
        output = run_tests()

        state.reproduction = {
            "reproduced": "failed" in output.lower(),
            "repro_steps": "Run: pytest sample_repo",
            "artifact": "sample_repo/test_app.py",
            "summary": "Test fails due to ZeroDivisionError"
        }

        print("[ReproductionAgent] Completed")
        return state

In [74]:
class FixAgent:
    def run(self, state):
        best = max(state.hypotheses, key=lambda x: x["confidence"])

        state.fix = {
            "root_cause": "The application fails due to lack of input validation in the divide function, causing a ZeroDivisionError when the denominator is zero.",
            "patch_plan": {
                "approach": "Introduce input validation before division",
                "code_fix": "if b == 0: return 0",
                "files": ["sample_repo/app.py"],
                "risk": "Behavior change if exception was expected"
            }
        }

        print("[FixAgent] Completed")
        return state

In [75]:
class CriticAgent:
    def run(self, state):
        state.review = {
            "confidence": 0.93,
            "checks": {
                "repro_valid": state.reproduction["reproduced"],
                "hypothesis_valid": len(state.hypotheses) > 0,
                "fix_safe": True
            }
        }
        print("[CriticAgent] Completed")
        return state

In [76]:
class AdvancedOrchestrator:
    def __init__(self):
        self.triage = TriageAgent()
        self.log = LogAgent()
        self.hypothesis = HypothesisAgent()
        self.repro = ReproductionAgent()
        self.fix = FixAgent()
        self.critic = CriticAgent()

    def run(self):
        state = DebugState()

        with open("inputs/bug_report.txt") as f:
            bug_report = f.read()

        with open("inputs/logs.txt") as f:
            logs = f.read()

        # Flow
        state = self.triage.run(bug_report, state)
        state = self.log.run(logs, state)
        state = self.hypothesis.run(state)
        state = self.repro.run(state)
        state = self.fix.run(state)
        state = self.critic.run(state)

        # FINAL REQUIRED OUTPUT
        result = {
            "bug_summary": state.bug_summary,

            "evidence": state.logs["errors"],

            "log_analysis": state.logs,
            "hypotheses": state.hypotheses,

            "repro_steps": state.reproduction["repro_steps"],
            "repro_artifact": state.reproduction["artifact"],

            "root_cause": state.fix["root_cause"],
            "patch_plan": state.fix["patch_plan"],

            "validation_plan": [
                "Run pytest sample_repo and confirm issue is resolved",
                "Add test case for divide(10, 0)",
                "Run regression tests"
            ],

            "confidence": state.review["confidence"],

            "open_questions": [
                "Should division by zero return 0 or raise custom exception?",
                "Are similar bugs present elsewhere?"
            ],

            "timestamp": str(datetime.now())
        }

        # Save output
        with open("outputs/result.json", "w") as f:
            json.dump(result, f, indent=4)

        print("\n FINAL OUTPUT GENERATED (submission-ready)")
        return result

In [77]:
system = AdvancedOrchestrator()
output = system.run()

output

[TriageAgent] Completed
[LogAgent] Completed
[HypothesisAgent] Completed
[ReproductionAgent] Completed
[FixAgent] Completed
[CriticAgent] Completed

 FINAL OUTPUT GENERATED (submission-ready)


{'bug_summary': {'error_type': 'ZeroDivisionError',
  'severity': 'Critical',
  'priority': 'High',
  'suspected_area': 'Arithmetic operation',
  'description': 'Crash due to invalid division operation'},
 'evidence': ['ERROR ZeroDivisionError: division by zero',
  'Traceback (most recent call last):'],
 'log_analysis': {'errors': ['ERROR ZeroDivisionError: division by zero',
   'Traceback (most recent call last):'],
  'error_count': 2,
  'pattern': 'ZeroDivisionError detected'},
 'hypotheses': [{'cause': 'Division by zero', 'confidence': 0.9},
  {'cause': 'Invalid input handling', 'confidence': 0.7}],
 'repro_steps': 'Run: pytest sample_repo',
 'repro_artifact': 'sample_repo/test_app.py',
 'root_cause': 'The application fails due to lack of input validation in the divide function, causing a ZeroDivisionError when the denominator is zero.',
 'patch_plan': {'approach': 'Introduce input validation before division',
  'code_fix': 'if b == 0: return 0',
  'files': ['sample_repo/app.py'],
 